# Cloud Cost Optimizer — Exploratory Data Analysis

This notebook walks through the synthetic dataset, visualises cost and usage patterns, and demonstrates each ML model in isolation.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from data.mock_generator import generate_dataset
from features.engineer import compute_resource_features, compute_daily_aggregates
from models.anomaly_detector import AnomalyDetector
from models.recommender import RecommendationEngine

plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 4)})
print('Libraries loaded.')

## 1  Generate the dataset

In [ ]:
df = generate_dataset(num_instances=50, days=90)
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.describe()

## 2  Daily total cost over time

In [ ]:
daily = compute_daily_aggregates(df)

fig, ax = plt.subplots()
ax.plot(daily['ds'], daily['y'], linewidth=1.5, color='steelblue')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
fig.autofmt_xdate()
ax.set(title='Total daily cost — all 50 instances', ylabel='USD', xlabel='Date')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.show()

## 3  Cost by environment

In [ ]:
env_daily = df.groupby(['date', 'environment'])['daily_cost_usd'].sum().reset_index()

fig, ax = plt.subplots()
colors = {'prod': '#3b82f6', 'dev': '#f97316', 'staging': '#8b5cf6'}
for env, grp in env_daily.groupby('environment'):
    ax.plot(grp['date'], grp['daily_cost_usd'], label=env, linewidth=1.5, color=colors.get(env))

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
fig.autofmt_xdate()
ax.set(title='Daily cost by environment', ylabel='USD')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 4  CPU utilisation distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, env in zip(axes, ['prod', 'dev', 'staging']):
    subset = df[df['environment'] == env]['cpu_utilization_avg']
    ax.hist(subset, bins=30, color=colors.get(env, 'gray'), edgecolor='white', alpha=0.85)
    ax.set(title=f'CPU — {env}', xlabel='CPU %', ylabel='Days')
    ax.axvline(subset.mean(), color='black', linewidth=1.5, linestyle='--', label=f'Mean {subset.mean():.1f}%')
    ax.legend(fontsize=9)
plt.suptitle('CPU utilisation by environment', y=1.02)
plt.tight_layout()
plt.show()

## 5  Feature engineering

In [ ]:
features = compute_resource_features(df)
print(f'Feature matrix shape: {features.shape}')
features[['resource_id', 'cpu_mean', 'cpu_waste_ratio', 'idle_day_pct', 'cost_mean_usd', 'environment']].head(10)

## 6  Anomaly detection

In [ ]:
detector = AnomalyDetector(contamination=0.05).fit(df)
anomalies = detector.predict(df)
print(f'Detected {len(anomalies)} anomalies')

anom_df = pd.DataFrame([
    {'date': a.date, 'resource_id': a.resource_id,
     'actual_cost': a.actual_cost, 'severity': a.severity, 'severity_label': a.severity_label}
    for a in anomalies
])
anom_df['date'] = pd.to_datetime(anom_df['date'])
anom_df.head(10)

In [ ]:
# Overlay anomalies on the daily cost chart
fig, ax = plt.subplots()
ax.plot(daily['ds'], daily['y'], linewidth=1, color='steelblue', label='Daily total cost')

if len(anom_df):
    anom_daily = anom_df.groupby('date')['actual_cost'].sum()
    ax.scatter(anom_daily.index, anom_daily.values,
               color='red', zorder=5, s=40, label='Anomaly day', alpha=0.7)

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
fig.autofmt_xdate()
ax.set(title='Daily cost with anomaly overlay', ylabel='USD')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

## 7  Recommendation engine

In [ ]:
engine = RecommendationEngine().fit(features)
recs   = engine.recommend(features)

recs_df = pd.DataFrame([
    {'resource_id': r.resource_id, 'action': r.action,
     'savings_usd': r.estimated_monthly_savings_usd,
     'confidence': r.confidence_score, 'risk': r.risk_level,
     'environment': r.environment}
    for r in recs
])
print(f'{len(recs_df)} recommendations generated')
recs_df.head(10)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Action distribution
recs_df['action'].value_counts().plot.bar(ax=ax1, color='steelblue', edgecolor='white')
ax1.set(title='Recommendations by action', ylabel='Count', xlabel='')
ax1.tick_params(axis='x', rotation=30)

# Savings by action
recs_df.groupby('action')['savings_usd'].sum().sort_values().plot.barh(ax=ax2, color='seagreen', edgecolor='white')
ax2.set(title='Total potential savings by action', xlabel='USD / month')
ax2.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

plt.tight_layout()
plt.show()

total = recs_df['savings_usd'].sum()
print(f'Total potential monthly savings: ${total:,.2f}')

## 8  Cost forecast (Prophet)

> This cell requires `prophet` to be installed (`pip install prophet`).

In [ ]:
try:
    from models.forecaster import CostForecaster

    forecaster = CostForecaster().fit(daily)
    forecast_30 = forecaster.forecast(horizon_days=30)

    fc_df = pd.DataFrame([{'ds': p.ds, 'yhat': p.yhat,
                            'lower': p.yhat_lower, 'upper': p.yhat_upper}
                           for p in forecast_30])
    fc_df['ds'] = pd.to_datetime(fc_df['ds'])

    fig, ax = plt.subplots()
    ax.plot(daily['ds'], daily['y'], color='steelblue', linewidth=1.5, label='Historical')
    ax.plot(fc_df['ds'], fc_df['yhat'], color='orange', linewidth=2, label='Forecast')
    ax.fill_between(fc_df['ds'], fc_df['lower'], fc_df['upper'],
                    color='orange', alpha=0.2, label='90 % CI')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    fig.autofmt_xdate()
    ax.set(title='30-day cost forecast (Prophet)', ylabel='USD')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print('Prophet not installed — run: pip install prophet')